# Loan Underwriting - Multi-Agent Demo
Manager-worker delegation on Semantic Kernel + Groq, with long-term memory (SQLite), bank comparison and advisory.

Bank data is **illustrative sample data (fictional banks)**, not live rates.

In [19]:
# !pip install -q semantic-kernel openai
import os, sys
sys.path.insert(0, '.')
# Set GROQ_API_KEY in the terminal (export GROQ_API_KEY=...). Never put the key in this file.

In [20]:
from run_demo import CASES
from agents.manager import UnderwritingManager
from state import print_trace
manager = UnderwritingManager()

In [21]:
# Sanction + personal-loan bank comparison
rec = await manager.underwrite(CASES['sanction'])
print_trace(rec)

[UnderwritingManager ] received_application         {"chars": 124}
[UnderwritingManager ] delegate                     {"to": "DocumentAgent"}
[DocumentAgent       ] tool:extract_fields          {"args": {"text": "Applicant: Ravi Kumar. Net monthly income Rs 60,000. Existing EMIs Rs 5,000. Requesting a loan of Rs 5,00,000 for 36 months."}, "re
[DocumentAgent       ] tool:validate_income_present {"args": {"income": 60000.0}, "result": true, "via": "kernel"}
[UnderwritingManager ] delegate                     {"to": "EmiAgent"}
[EmiAgent            ] tool:compute_emi             {"args": {"principal": 500000.0, "annual_rate": 10.5, "months": 36}, "result": 16251.22, "via": "kernel"}
[EmiAgent            ] tool:total_interest          {"args": {"emi": 16251.22, "months": 36, "principal": 500000.0}, "result": 85043.92, "via": "kernel"}
[UnderwritingManager ] delegate                     {"to": "AffordabilityAgent"}
[AffordabilityAgent  ] tool:emi_to_income_ratio     {"args": {"emi": 16251.

In [22]:
# Decline (unaffordable) -> banks not eligible + counter-offer
rec = await manager.underwrite(CASES['decline (unaffordable)'])
print_trace(rec)

[UnderwritingManager ] received_application         {"chars": 125}
[UnderwritingManager ] delegate                     {"to": "DocumentAgent"}
[DocumentAgent       ] tool:extract_fields          {"args": {"text": "Applicant: Neha Singh. Net monthly income Rs 30,000. Existing EMIs Rs 5,000. Requesting a loan of Rs 10,00,000 for 24 months."}, "r
[DocumentAgent       ] tool:validate_income_present {"args": {"income": 30000.0}, "result": true, "via": "kernel"}
[UnderwritingManager ] delegate                     {"to": "EmiAgent"}
[EmiAgent            ] tool:compute_emi             {"args": {"principal": 1000000.0, "annual_rate": 10.5, "months": 24}, "result": 46376.04, "via": "kernel"}
[EmiAgent            ] tool:total_interest          {"args": {"emi": 46376.04, "months": 24, "principal": 1000000.0}, "result": 113024.96, "via": "kernel"}
[UnderwritingManager ] delegate                     {"to": "AffordabilityAgent"}
[AffordabilityAgent  ] tool:emi_to_income_ratio     {"args": {"emi": 463

In [23]:
# Home loan: eligibility criteria, side-by-side comparison, recommendation
rec = await manager.underwrite(CASES['home loan comparison'])
print_trace(rec)

[UnderwritingManager ] received_application         {"chars": 107}
[UnderwritingManager ] delegate                     {"to": "DocumentAgent"}
[DocumentAgent       ] tool:extract_fields          {"args": {"text": "Applicant: Priya Nair. Net monthly income Rs 1,00,000. Requesting a home loan of Rs 30,00,000 for 20 years."}, "result": {"name": "
[DocumentAgent       ] tool:validate_income_present {"args": {"income": 100000.0}, "result": true, "via": "kernel"}
[UnderwritingManager ] delegate                     {"to": "EmiAgent"}
[EmiAgent            ] tool:compute_emi             {"args": {"principal": 3000000.0, "annual_rate": 10.5, "months": 240}, "result": 29951.4, "via": "kernel"}
[EmiAgent            ] tool:total_interest          {"args": {"emi": 29951.4, "months": 240, "principal": 3000000.0}, "result": 4188336.0, "via": "kernel"}
[UnderwritingManager ] delegate                     {"to": "AffordabilityAgent"}
[AffordabilityAgent  ] tool:emi_to_income_ratio     {"args": {"emi": 29

In [24]:
# Same applicant again: recalled from long-term memory
rec = await manager.underwrite(CASES['returning applicant (long-term memory)'])
print_trace(rec)

[UnderwritingManager ] received_application         {"chars": 100}
[UnderwritingManager ] delegate                     {"to": "DocumentAgent"}
[DocumentAgent       ] tool:extract_fields          {"args": {"text": "Applicant: Ravi Kumar. Net monthly income Rs 60,000. Requesting a loan of Rs 1,00,000 for 12 months."}, "result": {"name": "Ravi Ku
[DocumentAgent       ] tool:validate_income_present {"args": {"income": 60000.0}, "result": true, "via": "kernel"}
[UnderwritingManager ] delegate                     {"to": "EmiAgent"}
[EmiAgent            ] tool:compute_emi             {"args": {"principal": 100000.0, "annual_rate": 10.5, "months": 12}, "result": 8814.86, "via": "kernel"}
[EmiAgent            ] tool:total_interest          {"args": {"emi": 8814.86, "months": 12, "principal": 100000.0}, "result": 5778.32, "via": "kernel"}
[UnderwritingManager ] delegate                     {"to": "AffordabilityAgent"}
[AffordabilityAgent  ] tool:emi_to_income_ratio     {"args": {"emi": 8814.86, 

In [25]:
rec = await manager.underwrite(CASES['refer (income missing)'])
print_trace(rec)

[UnderwritingManager ] received_application         {"chars": 94}
[UnderwritingManager ] delegate                     {"to": "DocumentAgent"}
[DocumentAgent       ] tool:extract_fields          {"args": {"text": "Applicant: Meera Shah. Existing EMIs Rs 3,000. Requesting a loan of Rs 3,00,000 for 24 months."}, "result": {"name": "Meera Shah", 
[DocumentAgent       ] tool:validate_income_present {"args": {"income": 0.0}, "result": false, "via": "kernel"}
[UnderwritingManager ] halt                         {"missing": ["income"]}
[UnderwritingManager ] tool:decide_rule             {"args": {"income_ok": false, "affordable": false, "defaults": 0, "missed": 0}, "result": {"decision": "refer", "reasons": ["required data missing (e.
[UnderwritingManager ] tool:format_justification    {"args": {"decision": "refer", "reasons": "required data missing (e.g. income) - not assumed, referred to human underwriter", "emi": 0.0, "ratio": 0.0
[UnderwritingManager ] tool:remember_decision       {"args": 

In [26]:
# What long-term memory holds for Ravi Kumar
from memory import get_profile
get_profile('Ravi Kumar')

{'found': True,
 'defaults': 0,
 'missed_payments_12m': 0,
 'active_loans': 1,
 'past_applications': 6,
 'past_declines': 0,
 'past_sanctions': 6,
 'last_decision': 'sanction'}

In [27]:
print(''.join(open('run_log.jsonl').readlines()[-8:]))

{"ts": 1795362641.349, "agent": "UnderwritingManager", "event": "delegate", "data": {"to": "DocumentAgent"}}
{"ts": 1795362641.349, "agent": "DocumentAgent", "event": "tool:extract_fields", "data": {"args": {"text": "Applicant: Meera Shah. Existing EMIs Rs 3,000. Requesting a loan of Rs 3,00,000 for 24 months."}, "result": {"name": "Meera Shah", "income": null, "existing_emis": 3000.0, "amount": 300000.0, "tenure_months": 24, "annual_rate": null, "loan_type": "personal"}, "via": "kernel"}}
{"ts": 1795362642.14, "agent": "DocumentAgent", "event": "tool:validate_income_present", "data": {"args": {"income": 0.0}, "result": false, "via": "kernel"}}
{"ts": 1795362642.141, "agent": "UnderwritingManager", "event": "halt", "data": {"missing": ["income"]}}
{"ts": 1795362642.141, "agent": "UnderwritingManager", "event": "tool:decide_rule", "data": {"args": {"income_ok": false, "affordable": false, "defaults": 0, "missed": 0}, "result": {"decision": "refer", "reasons": ["required data missing (e.